# Імпорт необхідних бібліотек

In [ ]:
import warnings
from sys import getsizeof

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ml_homework.eda import age_group as age_cat
from ml_homework.eda import (
    iqr_bounds,
    null_summary,
    summary_table,
    years_from_days,
)
from ml_homework.eda import (
    upper_outlier_bound as outlier_range,
)
from ml_homework.paths import PROCESSED_DATA_DIR, RAW_DATA_DIR
from ml_homework.visualization import (
    boxplots_for_columns,
    category_counts_by_hue,
    correlation_heatmap,
    numeric_vs_categorical_analysis,
    rotated_countplot,
)
from ml_homework.visualization import (
    compare_boxplots as bi_boxplot,
)
from ml_homework.visualization import (
    compare_category_counts as bi_countplot_target,
)
from ml_homework.visualization import (
    compare_kde_without_upper_outliers as kde_no_outliers,
)
from ml_homework.visualization import (
    compare_scatter_without_outliers as plot_scatter_without_outliers_auto_bounds,
)
from ml_homework.visualization import (
    distribution_boxplot as dist_box,
)

In [ ]:
pd.set_option("display.max.rows", 130)
pd.set_option("display.max.columns", 130)
pd.set_option("float_format", "{:.2f}".format)

Зчитуємо дані.

In [ ]:
df = pd.read_csv(RAW_DATA_DIR / "credit/application_data.csv.zip")

In [ ]:
# Огляд декількох записів з датафрейму
df.head()

# Перевірка структури даних

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
df.shape

Маємо ~307k рядків та 122 колонки.

## Статистичний звіт для числових змінних

In [ ]:
df.describe()

# Аналіз категоріальних змінних

In [ ]:
df.select_dtypes(include="object").columns

In [ ]:
# Перевірка кількості категоріальних змінних
len(df.select_dtypes(include="object").columns)

Дані містять 16 `categorical` змінних

# Аналіз числових змінних

In [ ]:
number_df = df.select_dtypes(include="number")

In [ ]:
number_df.columns

In [ ]:
# Перевірка кількості числових змінних
len(number_df.columns)

Дані міcтять 106 `numerical` змінних

In [ ]:
number_df.head()

# Робота з некоректними типами даних

Перевірка, чи немає у нас стовпця з неправильним типом даних

In [ ]:
df.dtypes

Дивлячись на дані та відповідні їм типи даних, можна змінити

1.   Елемент списку
2.   Елемент списку

тип стовпчика SKU.

In [ ]:
df["SK_ID_CURR"] = df["SK_ID_CURR"].astype("str")

Також ми можемо змінити всі стовпці `flag` на тип даних, який є більш економний для зберігання.

Ось скільки пам'яті в Мб займають наші дані зараз.

In [ ]:
df.memory_usage().sum() / 1024 / 1024

Отже, ми можемо потенційно заощадити 57 Мб пам'яті! І трансформувати наш фрейм даних буде простіше. Давайте змінимо тип.


In [ ]:
flag_cols = ["flag" in col.lower() for col in df.columns]

In [ ]:
df[df.columns[flag_cols]].head()

In [ ]:
df[df.columns[flag_cols]].nunique()

Типи даних в pandas взяті з numpy, і ось тут список всіх типів даних в numpy:

https://numpy.org/doc/stable/user/basics.types.html

Кожен тип даних займає певну кількість байт у пам'яті. Давайте з'ясуємо, скільки займає 1 тип int8 та 1 тип int64 і скільки пам'яті ми заощадимо, якщо змінимо тип.

In [ ]:
getsizeof(np.int64(1))

In [ ]:
getsizeof(np.int8(1))

In [ ]:
7 * df.shape[0] * sum(flag_cols) / 1024 / 1024

In [ ]:
str_flag_cols = ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]

In [ ]:
for col in str_flag_cols:
    df[col] = np.where(df[col] == "Y", 1, 0)

In [ ]:
df[str_flag_cols].nunique()

In [ ]:
df[df.columns[flag_cols]] = df[df.columns[flag_cols]].astype("int8")

In [ ]:
df.memory_usage().sum() / 1024 / 1024

# Робота з пропущеними значеннями

Найпростіше емпіричне правило для опрацювання пропущених значень: якщо пропущених значень більше за 40% - видаляємо колонку, якщо менше за 40% - аналізуємо, як можна заповнити і чи треба.

Зазвичай, якщо відсоток пропущених даних більший за 10 і немає чіткої (яка значно виділяється) моди (найчастішого) значення в даних, то лишаємо дані як є до використання методів машинного навчання. Далі якщо метод вимагає заповення пропущених значень, можемо експериментувати із заповеннями. Також хорошою практикою є створити окрему колонку-флаг (0/1), яка вказує, де були пропущені значення - це буде додаткова ознака для моделі.


Перевіримо, чи немає нульових значень в нашому наборі даних

In [ ]:
df.isnull().values.any()

Порахуємо загальну кількість нульових значень в наборі даних

In [ ]:
df.isnull().values.sum()

Сформуємо список із стовпців з нульовими значеннями

In [ ]:
df.columns[df.isnull().any()]

In [ ]:
len(df.columns[df.isnull().any()])

Усього `67` стовпців мають одне або більше NULL-значень в даних

## Кількість та відсоток пропущених значень у стовпцях

In [ ]:
null_df = null_summary(df)

In [ ]:
null_df.sort_values(by="null_percentage", ascending=False)

## Видалення стовпців з NULL значеннями > 40%

Сформуємо список стовпців з NULL значеннями > 40% у список. Ми видалимо ці стовпці з датафрейму, оскільки в них занадто багато пропущених значень.

In [ ]:
columns_to_be_deleted = null_df[null_df["null_percentage"] > 40].column_name.to_list()

In [ ]:
len(columns_to_be_deleted)

Всього потрібно видалити `49` стовпців. Видалення їх з основного датафрейму **`df`**

In [ ]:
df.drop(columns=columns_to_be_deleted, inplace=True)

Перевірка підрахунку стовпців після видалення. Мало б залишитись лише `73` стовпці

In [ ]:
df.shape

## Перевірка стовпців з NULL значеннями < 40%

Створення датафрейму `null_df_under40` зі стовпцями, де відсоток пропущених значеннь менше 40%

In [ ]:
null_df_under40 = null_df[null_df["null_percentage"] < 40]

In [ ]:
null_df_under40.sort_values(by="null_percentage", ascending=False)

Опрацюємо кожну з колонок.

### Аналіз стовпця `OCCUPATION_TYPE`

- нульові значення = 31.35%

In [ ]:
df["OCCUPATION_TYPE"].value_counts()

Заміна NULL-значень на категорією `Unknown`

In [ ]:
df["OCCUPATION_TYPE"].fillna(value="Unknown", inplace=True)

In [ ]:
rotated_countplot(df, "OCCUPATION_TYPE")
plt.show()

**Спостереження**
- Якщо поглянути на графік, то найбільшу кількість заявників на кредит мають `Laborers`
- Для імпутації краще залишити дані як є (пропущені значення становлять 31,35%) і не проводити імпутацію за якоюсь константою, як-от мода або мін/макс медіана, якщо це числовий стовпчик, оскільки це може викривити дані в подальших розрахунках.

Існує також можливість імпутації за допомогою більш "розумних" методів, але ми вивчимо їх пізніше, а зараз ми робимо те, що можемо, за допомогою деяких найпростіших перетворень.

### Аналіз стовпця `EXT_SOURCE_3`

- пропущені значення = 19.83%

In [ ]:
df.EXT_SOURCE_3.value_counts().head()

In [ ]:
sns.boxplot(data=df, y="EXT_SOURCE_3")
plt.show()

Отримання процентильних значень для `EXT_SOURCE_3`

In [ ]:
df.EXT_SOURCE_3.quantile(q=[0.25, 0.5, 0.75, 1])

Найбільш повторюване значення в `EXT_SOURCE_3

In [ ]:
df.EXT_SOURCE_3.mode()[0]

Перевірка середнього значення `EXT_SOURCE_3`

In [ ]:
df.EXT_SOURCE_3.mean()

**Спостереження**
-  Дивлячись на діагараму розмаху, медіана становить 0,535276
-  Найчастіше повторюване значення - 0,74630
-  Середнє значення - 0,51085
-  Хоча середнє та медіана є ближчими і можуть бути використані для імпутації, оскільки відсутнє відсоткове значення є більшим (19,83%), краще залишити дані як є і не проводити імпутації. Якщо метод вимагає імпутації, ми можемо імпутувати дані за допомогою медіани і створити ще один стовпчик, в якому буде позначено, які значення були імпутовані.

# ДЗ 1. Аналіз стовпців `AMT_REQ_CREDIT_BUREAU` з пропущеними значеннями

Виведіть середнє, моду, медіану та відсоток відсутніх даних для настуних колонок:

- `AMT_REQ_CREDIT_BUREAU_YEAR`
-	`AMT_REQ_CREDIT_BUREAU_MON`
-	`AMT_REQ_CREDIT_BUREAU_WEEK`

На основі виведених даних напишіть висновок про те, чи варто заповнювати пусті значення і якщо так, то яким чином?

In [ ]:
bureau_columns = [
    "AMT_REQ_CREDIT_BUREAU_YEAR",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
]
df_bureau_columns = df[bureau_columns]

result = pd.DataFrame(
    {
        "mean": df_bureau_columns.mean(),
        "mode": df_bureau_columns.mode().iloc[0],
        "median": df_bureau_columns.median(),
        "null_percentage": df_bureau_columns.isnull().mean() * 100,
    }
)
result

In [ ]:
plt.figure(figsize=(10, 5))
df_bureau_columns.boxplot()

plt.title("Розподіл запитів до бюро")
plt.ylabel("Кількість запитів")
plt.show()

Проаналізувавши дані про розподіл запитів до бюро за рік/місяць/та тиждень, ми можемо побачити що в цих колонках в нас є 13.5% пропущених значень.
Оскільки на боксплоті можна побачити дуже багато викидів, то я б не запонювала порожні значення середнім, оскільки це буде завищене значення.
А ось мода і медіана в нас в розподілі за тиждень і місяць однакові і рівні 0(можна заповнити пропуски або модою або медіаною), а за рік - медіана дорівнює 1, а мода - 0(тут краще використати медіану, оскільки вона краще відображає типове значення і є стійкою до викидів),
а створити ще один стовпчик, в якому буде позначено, які значення були імпутовані.

In [ ]:
for column in bureau_columns:
    df[f"FLAG_{column}_MISSING"] = df[column].isnull().astype(int)

df[bureau_columns] = df[bureau_columns].fillna(df[bureau_columns].median())


flag_columns = [f"FLAG_{column}_MISSING" for column in bureau_columns]

flag_columns

## Перевірка стовпців зі значеннями NULL > 0% та < 1%

Створення датафрейму `null_df_under1` з відсотком пропущених значень > 0% та < 1% у кожному стовпці

In [ ]:
null_df_under1 = null_df[
    (null_df["null_percentage"] > 0) & (null_df["null_percentage"] < 1)
]

In [ ]:
null_df_under1.sort_values(by="null_percentage", ascending=False)

### Аналіз стовпця `NAME_TYPE_SUITE`

In [ ]:
null_df_under1[null_df_under1.column_name == "NAME_TYPE_SUITE"]

In [ ]:
df["NAME_TYPE_SUITE"].value_counts()

In [ ]:
rotated_countplot(df, "NAME_TYPE_SUITE")
plt.show()

**Спостереження**
-   Дивлячись на графік, категорія `Unaccompanied` має найбільшу кількість заявників на отримання кредиту. Отже, більшість позичальників наважуються звертатися за кредитом без супроводу.
- Ми можемо продовжити імпутацію `Unaccompanied` в датафреймі, але краше надати перевагу другому варіанту.
- Ми також могли б імпутувати дані зі значенням `NA`, оскільки ця колонка є категоричною.
- Аналогічно, якщо в колонці не вистачає менше 1% даних, ми можемо її опустити. Але якщо ми вилучимо всі дані в усіх стовпчиках, де пропущено <=1% даних, ми можемо вилучити занадто багато даних. Тому я здебільшого зберігаю дані настільки, наскільки це можливо.

In [ ]:
df["NAME_TYPE_SUITE"].fillna("NA", inplace=True)

### Аналіз стовпця `OBS_30_CNT_SOCIAL_CIRCLE`

In [ ]:
null_df_under1[null_df_under1.column_name == "OBS_30_CNT_SOCIAL_CIRCLE"]

In [ ]:
df.OBS_30_CNT_SOCIAL_CIRCLE.value_counts().head(10)

In [ ]:
sns.boxplot(data=df, y="OBS_30_CNT_SOCIAL_CIRCLE")
plt.show()

Розрахунок перцентилів `OBS_30_CNT_SOCIAL_CIRCLE`

In [ ]:
df.OBS_30_CNT_SOCIAL_CIRCLE.quantile(q=[0.25, 0.5, 0.75, 1])

Найбільш повторюване значення в `OBS_30_CNT_SOCIAL_CIRCLE`

In [ ]:
df.OBS_30_CNT_SOCIAL_CIRCLE.mode()[0]

Середнє значення `OBS_30_CNT_SOCIAL_CIRCLE`

In [ ]:
df.OBS_30_CNT_SOCIAL_CIRCLE.mean()

**Спостереження**
- Дивлячись на діаграму розмаху, медіана дорівнює 0.0
- Найчастіше повторюване значення - 0.0
- Середнє значення - 1,4222
- Є два викидні значення на рівні 50 та 350.
- Медіана і мода близькі (з огляду на діапазон даних у цьому стовпчику) і можуть бути використані для імпутації. Це не призведе до зміщення, оскільки відсоток пропущених значень невеликий (0,33%)

In [ ]:
df["OBS_30_CNT_SOCIAL_CIRCLE"].fillna(
    df["OBS_30_CNT_SOCIAL_CIRCLE"].median(), inplace=True
)

# ДЗ 2. Аналіз і заповнення пустих значень у колонках з малим відсотком пропущених

За прикладом вище проведіть аналіз пропущених значень в колонках
- EXT_SOURCE_2
- AMT_GOODS_PRICE

Для швидшого аналізу рекомендую написати фукнцію, яку Ви зможете викликати для кожної з колонок.

Зробіть висновок, що робити з пропущеними значеннями в кожному випадку і виконанайте ту дію, яку зазначили.

In [ ]:
columns_to_analyze = ["EXT_SOURCE_2", "AMT_GOODS_PRICE"]

summary_table(df, null_df_under1, columns_to_analyze)

In [ ]:
boxplots_for_columns(df, columns_to_analyze)

Для колонки ***EXT_SOURCE_2*** показники такі:

**null_percentage** = 0.21%
**mode** = 0.29
**mean** = 0.51
**median** = 0.57

Оскільки відсоток пропущених значень маленький, викиди відсутні і значення середнього та медіани не дуже сильно відрізняються, то можна пропуски заповнити одним із цих значень.



In [ ]:
df["EXT_SOURCE_2"] = df["EXT_SOURCE_2"].fillna(df["EXT_SOURCE_2"].median())

In [ ]:
df["EXT_SOURCE_2"].isnull().sum()

Для колонки ***AMT_GOODS_PRICE*** показники такі:

**null_percentage** = 0.09%
**mode** = 450000
**mean** = 538396.21
**median** = 450000

Тут також відсоток пропущених значень маленький (0.09%), але на боксплоті ми бачимо багато викидів у верхній частині, що відповідно впливає на середнє значення та тягне його вгору, тому ми і бачимо що mean у нас більше за median та mode, які однакові. В даному випадку я краще б заповниа пропуски медіаною, яка є стійкішою до викидів.



In [ ]:
df["AMT_GOODS_PRICE"] = df["AMT_GOODS_PRICE"].fillna(df["AMT_GOODS_PRICE"].median())

In [ ]:
df["AMT_GOODS_PRICE"].isnull().sum()

# Робота з неправильними/невідомими значеннями даних

### Аналіз стовпця `CODE_GENDER`

Перевірка діапазону значень

In [ ]:
df["CODE_GENDER"].value_counts()

Стать має бути тільки чоловіча або жіноча. Значення `XNA` може вказувати на те, що значення не було надано заявником або пропущено кредитним спеціалістом, який перевіряє заявку

In [ ]:
df[df["CODE_GENDER"] == "XNA"]

Оскільки дані виглядають достовірними, ми перевіримо можливість застосування методу імпутації.
- Заявників-жінок удвічі більше, ніж заявників-чоловіків
- Отже, ми прирівняємо `CODE_GENDER` до 'F'

In [ ]:
df["CODE_GENDER"] = df["CODE_GENDER"].apply(lambda x: "F" if x == "XNA" else x)

Перевірка, чи вилучено `XNA`

In [ ]:
df["CODE_GENDER"].value_counts()

### Аналіз стовпця `DAYS_BIRTH`

In [ ]:
df["DAYS_BIRTH"].value_counts().head()

Існує ~17K+ унікальних записів, всі з яких, схоже, мають від'ємні значення

In [ ]:
df["DAYS_BIRTH"].unique()

In [ ]:
df["DAYS_BIRTH"].nunique()

In [ ]:
df["DAYS_BIRTH"].describe()

Перетворення `Days Birth` на додатні дні

In [ ]:
df["DAYS_BIRTH"] = df["DAYS_BIRTH"].abs()

In [ ]:
df["DAYS_BIRTH"].value_counts()

Всі дні в `DAYS_BIRTH` мають додатні значення - це може бути зручніше для аналізу.

#### Створимо нову колонку `YEARS_BIRTH` для зручності аналізу

In [ ]:
df["YEARS_BIRTH"] = years_from_days(df["DAYS_BIRTH"])

### Аналіз стовпця `NAME_FAMILY_STATUS`

Перевірка діапазону значень

In [ ]:
df["NAME_FAMILY_STATUS"].value_counts()

Стать має бути тільки чоловіча або жіноча. Значення `Unknown` може означати, що значення не було вказане заявником або пропущене кредитним спеціалістом, який перевіряє заявку

In [ ]:
df[df["NAME_FAMILY_STATUS"] == "Unknown"]

In [ ]:
df["NAME_FAMILY_STATUS"].value_counts(normalize=True) * 100

Оскільки дані виглядають достовірними, ми продовжимо перевірку методу імпутації.
- Одружені заявники складають понад 63% заявників.
- Отже, ми прирівняємо `NAME_FAMILY_STATUS` до `Married`.

In [ ]:
df["NAME_FAMILY_STATUS"] = df["NAME_FAMILY_STATUS"].apply(
    lambda x: "Married" if x == "Unknown" else x
)

Перевірка чи вилучено `Unknown`

In [ ]:
df["NAME_FAMILY_STATUS"].value_counts()

### Аналіз стовпця`DAYS_EMPLOYED`

In [ ]:
df["DAYS_EMPLOYED"].value_counts().head()

In [ ]:
df["DAYS_EMPLOYED"].value_counts(normalize=True) * 100

In [ ]:
len(df[df["DAYS_EMPLOYED"] < 365243])

In [ ]:
df[df["DAYS_EMPLOYED"] < 365243].DAYS_EMPLOYED.value_counts()

In [ ]:
df["DAYS_EMPLOYED"].unique()

In [ ]:
df["DAYS_EMPLOYED"].nunique()

**Спостереження**
- Існує ~55K+ записів, для яких `DAYS_EMPLOYED` дорівнює 365243 дням
- Решта 252K+ записів мають від'ємне значення днів
- Існує 12 574 унікальних значень для `DAYS_EMPLOYED`

 - Колонка `DAYS_EMPLOYED` вказує на те, за скільки днів до подачі заявки особа почала поточну роботу, заявник/кредитний спеціаліст повинен ввести від'ємні значення, щоб вказати дні, що передують даті подачі заявки.<br>
 - Ми конвертуємо від'ємні значення в `DAYS_EMPLOYED` в додатні дні, щоб стандартизувати дні під час використання в розрахунках

In [ ]:
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].abs()

In [ ]:
df["DAYS_EMPLOYED"].value_counts().head()

Ми бачимо, що всі дні в `DAYS_EMPLOYED` мають додатні значення

**Для ~55K+ записів, для яких `DAYS_EMPLOYED` становить 365243 дні**
- Конвертуючи це в роки, ми отримуємо 1000 років, які фізично неможливо використати для працевлаштування заявника
- Це відповідає 18% даних і не може бути аномалією
- Вони можуть бути або пенсіонерами, або безробітними, і, дивлячись на дані, наш висновок є правильним

> Є два способи впоратися з цим
>> 1) Ми залишаємо дані такими, якими вони є, і враховуємо це під час аналізу АБО  <br>
>> 2) Ми розраховуємо середню кількість днів зайнятості без урахування цієї категорії та вписуємо її замість 365243 днів для пенсіонерів. <br>
>> Для безробітних кількість днів зайнятості може дорівнювати 0

*Примітка*
 - Під час розрахунків у цьому стовпчику ми повинні враховувати цей сценарій, оскільки інакше він спотворює наші результати

In [ ]:
df[df["DAYS_EMPLOYED"] == 365243].NAME_INCOME_TYPE.value_counts()

#### Створимо нову колонку `YEARS_EMPLOYED` для зручності аналізу

In [ ]:
df["YEARS_EMPLOYED"] = years_from_days(df["DAYS_EMPLOYED"])

### Аналіз стовпця `DAYS_REGISTRATION`

In [ ]:
df["DAYS_REGISTRATION"].value_counts().head()

In [ ]:
df["DAYS_REGISTRATION"].value_counts(normalize=True).head()

In [ ]:
df["DAYS_REGISTRATION"].unique()

In [ ]:
df["DAYS_REGISTRATION"].nunique()

Перетворення `DAYS_REGISTRATION` в додатні дні

In [ ]:
df["DAYS_REGISTRATION"] = df["DAYS_REGISTRATION"].abs()

In [ ]:
df["DAYS_REGISTRATION"].value_counts().head()

Всі дні в `DAYS_REGISTRATION` мають додатні значення

#### Створимо нову колонку `YEARS_REGISTRATION` для зручності аналізу

In [ ]:
df["YEARS_REGISTRATION"] = years_from_days(df["DAYS_REGISTRATION"])

### Аналіз стовпця `DAYS_ID_PUBLISH`

In [ ]:
df["DAYS_ID_PUBLISH"].value_counts().head()

In [ ]:
df["DAYS_ID_PUBLISH"].value_counts(normalize=True).head()

In [ ]:
df["DAYS_ID_PUBLISH"].unique()

In [ ]:
df["DAYS_ID_PUBLISH"].nunique()

Перетворення `DAYS_ID_PUBLISH` в додатні дні

In [ ]:
df["DAYS_ID_PUBLISH"] = df["DAYS_ID_PUBLISH"].abs()

In [ ]:
df["DAYS_ID_PUBLISH"].value_counts().head()

Всі дні в `DAYS_ID_PUBLISH` мають додатні значення

#### Створимо нову колонку `YEARS_ID_PUBLISH` для зручності аналізу

In [ ]:
df["YEARS_ID_PUBLISH"] = years_from_days(df["DAYS_ID_PUBLISH"])

### Аналіз стовпця `DAYS_LAST_PHONE_CHANGE`

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"].value_counts().head()

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"].value_counts(normalize=True).head()

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"].unique()

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"].nunique()

Перетворення `DAYS_LAST_PHONE_CHANGE` в додатні дні

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"] = df["DAYS_LAST_PHONE_CHANGE"].abs()

In [ ]:
df["DAYS_LAST_PHONE_CHANGE"].value_counts().head()

Всі дні в `DAYS_LAST_PHONE_CHANGE` мають додатні значення

#### Створення нової колонки `YEARS_LAST_PHONE_CHANGE` для зручності аналізу

In [ ]:
df["YEARS_LAST_PHONE_CHANGE"] = years_from_days(df["DAYS_LAST_PHONE_CHANGE"])

# Автоматичні функції

### Створення функції `age_cat` для категоризації `YEARS_BORNING`

### Одномірний (категоріальний vs цільовий) та двомірний (категоріальний vs категоріальний) аналіз (гістограма) - категоріальні стовпчики

### Обчислення мінімального та максимального діапазону відхилень для числових стовпців

# Робота з викидами для числових стовпців

### Аналіз стовпця`CNT_CHILDREN`

In [ ]:
df["CNT_CHILDREN"].value_counts().sort_values(ascending=False).head()

In [ ]:
(
    df["CNT_CHILDREN"].value_counts(normalize=True).sort_values(ascending=False) * 100
).head()

In [ ]:
dist_box(df, "CNT_CHILDREN")

Розрахуємо IQR (Inter Quartile range)

In [ ]:
min_value, max_value = iqr_bounds(df["CNT_CHILDREN"])
print(f"Мінімальне значення, до якого існують викиди: {min_value}")
print(f"Максимальне значення, після якого існують викиди: {max_value}")

Значення *до* (Q1 - 1,5 * IQR) та *після* (Q3 + 1,5 * IQR) є викидами

**Спостереження**
- Дивлячись на дані, ми бачимо, що кількість заявників, які мають більше 7 дітей, є дуже мінімальною (2 або 3 в кожній категорії)
- Крім того, дивлячись на дані для заявників з 10 дітьми, заявники мають лише 31 й 41 рік відповідно. Це виглядає як одиничний випадок й може розглядатися як відхилення від норми
- Як дистрибутивні, так і діаграми розмаху чітко показують, що значення, які перевищують значення 2.5, є відхиленнями від норми.

**Висновок**
- Заявники, які мають 3 або більше дітей, є випадками, що відхиляються від норми. Ми можемо надати спеціальний аналіз для цих випадків.

### Аналіз стовпця `AMT_INCOME_TOTAL`

In [ ]:
df["AMT_INCOME_TOTAL"].value_counts().sort_values(ascending=False).head()

In [ ]:
(
    df["AMT_INCOME_TOTAL"].value_counts(normalize=True).sort_values(ascending=False)
    * 100
).head()

In [ ]:
df["AMT_INCOME_TOTAL"].describe(percentiles=[0.75, 0.99, 0.999])

Побудуємо графік для `AMT_INCOME_TOTAL`

In [ ]:
dist_box(df, "AMT_INCOME_TOTAL")

- Графіки кінцевого результату дуже тонкі, й ми можемо спостерігати викид близько ~120 мільйонів.
- Давайте побудуємо графік, розглядаючи лише дохід нижче 99,9% значення, тобто 900 тисяч.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    plt.subplots(1, 2, figsize=(20, 8))

    plt.subplot(121)
    sns.histplot(df.loc[df["AMT_INCOME_TOTAL"] < 900000, "AMT_INCOME_TOTAL"], kde=True)
    pltname = "Distplot of " + "AMT_INCOME_TOTAL"
    plt.title(pltname)

    plt.subplot(122)
    sns.boxplot(df[df["AMT_INCOME_TOTAL"] < 900000].AMT_INCOME_TOTAL)
    pltname = "Boxplot of " + "AMT_INCOME_TOTAL"
    plt.title(pltname)

    plt.tight_layout(pad=4)
    plt.show()

Тепер ми можемо чітко побачити розподіл і діапазон даних на обох графіках.
 - Це означає, що значення понад 900 тис. грн доходу явно є викидами

In [ ]:
df[df["AMT_INCOME_TOTAL"] > 900000].head()

**Спостереження**
- Дивлячись на дані, ми бачимо, що дохід понад 900 тис. грн (99,9% значення) є викидом
- Як діаграма розподілу, так і діаграма розмаху чітко показують нам ту саму тенденцію

**Висновок**
- Апліканти з доходом понад 900 тис. грн (99,9% значення) є викидами

# ДЗ 3. Аналіз викидів

Подібно до прикладу, проаналізуйте на викиди колонок
- `CNT_FAM_MEMBERS`
- `AMT_ANNUITY`

Зазначте, які значення в них можна вважати викидами.

In [ ]:
df["CNT_FAM_MEMBERS"].value_counts()

In [ ]:
df["CNT_FAM_MEMBERS"].value_counts(normalize=True).sort_values(ascending=False) * 100

In [ ]:
df["CNT_FAM_MEMBERS"].describe(percentiles=[0.75, 0.99, 0.999])

In [ ]:
dist_box(df, "CNT_FAM_MEMBERS")

In [ ]:
max_fam_members_value = outlier_range(df, "CNT_FAM_MEMBERS")
max_fam_members_value

Проаналізувавши дані колонки CNT_FAM_MEMBERS, думаю, що значення, які вище 4.5 можна вважати викидами. Як діаграма розподілу, так і діаграма розмаху чітко показують нам ту саму тенденцію, і це підкріплюється максимальним значенням, після якого існують викиди, яке також дорювнює 4.5.




**AMT_ANNUITY**

In [ ]:
df["AMT_ANNUITY"].value_counts()

In [ ]:
df["AMT_ANNUITY"].value_counts(normalize=True).sort_values(ascending=False) * 100

In [ ]:
df["AMT_ANNUITY"].unique()

In [ ]:
df["AMT_ANNUITY"].nunique()

In [ ]:
df["AMT_ANNUITY"].describe(percentiles=[0.75, 0.99, 0.999])

In [ ]:
dist_box(df, "AMT_ANNUITY")

In [ ]:
dist_box(df[df["AMT_ANNUITY"] < 110047.50], "AMT_ANNUITY")

In [ ]:
max_amt_annuity_value = outlier_range(df, "AMT_ANNUITY")
max_amt_annuity_value

Проаналізувавши дані колонки AMT_ANNUITY, думаю, що значення, які вище 61704.0 можна вважати викидами. Як діаграма розподілу, так і діаграма розмаху чітко показують нам ту саму тенденцію, і це підкріплюється максимальним значенням, після якого існують викиди, яке також дорювнює 61704.0.

# Розбиття на біни безперервних колонок для аналізу

### Категоризація стовпця `AMT_GOODS_PRICE

In [ ]:
df["AMT_GOODS_PRICE"].value_counts().sort_values(ascending=False).head()

In [ ]:
(
    df["AMT_GOODS_PRICE"].value_counts(normalize=True).sort_values(ascending=False)
    * 100
).head()

Подивимось статистичний звіт для `AMT_GOODS_PRICE

In [ ]:
df["AMT_GOODS_PRICE"].describe(percentiles=[0.25, 0.75, 0.99, 0.9999])

Розподілимо значення в `AMT_GOODS_PRICE` на 5 бінів і створимо новий стовпець `AMT_GOODS_PRICE_CATEGORY`.

In [ ]:
df["AMT_GOODS_PRICE_CATEGORY"] = pd.cut(
    df["AMT_GOODS_PRICE"],
    bins=5,
    labels=["very low", "low", "medium", "high", "very high"],
)

Перевірка заповнення значень згідно з очікуванням

In [ ]:
df["AMT_GOODS_PRICE_CATEGORY"].value_counts()

### Категоризація стовпця `YEARS_BIRTH`

Ми будемо класифікувати `YEARS_BIRTH` замість `DAYS_BIRTH`, оскільки роки легше інтерпретувати, ніж дні

In [ ]:
df["YEARS_BIRTH"].value_counts().sort_values(ascending=False).head()

In [ ]:
(
    df["YEARS_BIRTH"].value_counts(normalize=True).sort_values(ascending=False) * 100
).head()

Подивимось статистичний звіт для  `YEARS_BIRTH`

In [ ]:
df["YEARS_BIRTH"].describe(percentiles=[0.25, 0.75, 0.99, 0.9999])

Категоризуймо значення з `YEARS_BIRTH` у новий стовпець `YEARS_BIRTH_CATEGORY`.

In [ ]:
df["YEARS_BIRTH_CATEGORY"] = df["YEARS_BIRTH"].apply(age_cat)

Перевірка заповнення значень згідно з очікуванням

In [ ]:
df["YEARS_BIRTH_CATEGORY"].value_counts().sort_values(ascending=False)

### Категоризація стовпця `YEARS_REGISTRATION`

Ми будемо класифікувати `YEARS_REGISTRATION` замість `DAYS_REGISTRATION`, оскільки роки легше інтерпретувати, ніж дні

In [ ]:
df["YEARS_REGISTRATION"].value_counts().sort_values(ascending=False).head()

In [ ]:
(
    df["YEARS_REGISTRATION"].value_counts(normalize=True).sort_values(ascending=False)
    * 100
).head()

Подивимось статистичний звіт для `YEARS_REGISTRATION`

In [ ]:
df["YEARS_REGISTRATION"].describe(percentiles=[0.25, 0.75, 0.99, 0.9999])

Категоризуймо значення з `YEARS_REGISTRATION` в новий стовпець `YEARS_REGISTRATION_CATEGORY`.

In [ ]:
df["YEARS_REGISTRATION_CATEGORY"] = df["YEARS_REGISTRATION"].apply(age_cat)

Перевірка заповнення значень згідно з очікуванням

In [ ]:
df["YEARS_REGISTRATION_CATEGORY"].value_counts().sort_values(ascending=False)

# Зберігання оновлених даних зі стисненням

In [ ]:
filename = PROCESSED_DATA_DIR / "credit/application_data_processed"
compression_options = dict(method="zip", archive_name=f"{filename}.csv")
df.to_csv(f"{filename}.zip", compression=compression_options, index=False)

# Перевірка дисбалансу для цільового стовпця `TARGET`

### Аналіз стовпця `TARGET`

In [ ]:
df["TARGET"].value_counts().sort_values(ascending=False)

In [ ]:
df["TARGET"].value_counts(normalize=True).sort_values(ascending=False) * 100

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x=df["TARGET"], data=df)
plt.title("Перевірка коефіцієнта дисбалансу змінної TARGET")
plt.show()

**Спостереження**

- Ми маємо дисбаланс у змінній `TARGET` на основі % спостережень
 - Значення `TARGET` 1 - це клієнт, який має труднощі з оплатою (прострочення платежу більше ніж на X днів принаймні за одним з перших Y платежів за кредитом). Це лише 8,07% даних
 - Значення `TARGET` 0 - всі інші випадки, окрім 1. Це 91,93% даних

# Розподіл даних на основі `TARGET`

### Створіть новий фрейм даних зі значенням `TARGET` 1
- Значення `TARGET` 1 представляє клієнта з платіжними труднощами (він/вона прострочив платіж більш ніж на X днів принаймні по одному з перших Y платежів за кредитом). Це лише 8,07% даних

In [ ]:
df1 = df[df["TARGET"] == 1]

In [ ]:
df1.TARGET.value_counts()

### Створити новий фрейм даних зі значенням `TARGET` 0
- Значення `TARGET` 0 представляє всі інші випадки, крім 1. Це 91.93% даних

In [ ]:
df0 = df[df["TARGET"] == 0]

In [ ]:
df0.TARGET.value_counts()

# Одномірний аналіз категоріальних змінних

### Аналіз стовпця `NAME_CONTRACT_TYPE`

In [ ]:
df["NAME_CONTRACT_TYPE"].value_counts().sort_values(ascending=False)

In [ ]:
category_counts_by_hue(df, "NAME_CONTRACT_TYPE", "TARGET")

**Спостереження**

- Дивлячись на гістограми, ми не бачимо суттєвих відмінностей у `NAME_CONTRACT_TYPE` між клієнтами, які мають труднощі з оплатою, та клієнтами, які платять вчасно

**Висновок**
- Колонка `NAME_CONTRACT_TYPE` не надає жодних переконливих доказів на користь клієнтів, які мають труднощі з оплатою АБО вчасно сплачують

# ДЗ 4. Одновимірний аналіз категоріальної колонки

За наведеним прикладом вище, проведіть аналіз залежності між категоріальною колонкою і цільовою для колонок
- CODE_GENDER
- FLAG_OWN_CAR

Опціонально (для тих, кому цікаво дослідити більше даних)
- NAME_INCOME_TYPE
- NAME_EDUCATION_TYPE
- NAME_FAMILY_STATUS


Чи є вплив цих змінних на цільову та який саме?

In [ ]:
df["CODE_GENDER"].value_counts()

In [ ]:
df["CODE_GENDER"].value_counts(normalize=True) * 100

In [ ]:
gender_target_rate = df.groupby("CODE_GENDER")["TARGET"].mean().mul(100).round(2)

gender_target_rate

In [ ]:
category_counts_by_hue(df, "CODE_GENDER", "TARGET")

Спостереження

Дивлячись на гістограми, можна побачити відмінності у розподілі CODE_GENDER між клієнтами, які мають труднощі з оплатою, та клієнтами, які платять вчасно.
Серед клієнтів, які платять вчасно, жінки становлять 66.6%, а чоловіки — 33.4%. Серед клієнтів, які мають труднощі з оплатою, жінки становлять 57.1%, а чоловіки — 42.9%. На перший погляд може здатися, що жінок серед проблемних клієнтів більше. Але це пов’язано з тим, що жінок загалом більше в датасеті.

Щоб правильно оцінити залежність між CODE_GENDER і TARGET, потрібно дивитися на частку проблемних клієнтів всередині кожної категорії. Серед жінок приблизно 7% мають труднощі з оплатою, а серед чоловіків — приблизно 10.1%.

Висновок

Чоловіки мають вищу частку клієнтів із труднощами з оплатою порівняно з жінками. Отже, колонка CODE_GENDER може бути корисною для аналізу ризику прострочення, оскільки розподіл TARGET відрізняється між категоріями.

In [ ]:
df["FLAG_OWN_CAR"].value_counts()

In [ ]:
df["FLAG_OWN_CAR"].value_counts(normalize=True) * 100

In [ ]:
gender_target_rate = df.groupby("FLAG_OWN_CAR")["TARGET"].mean().mul(100).round(2)

gender_target_rate

In [ ]:
category_counts_by_hue(df, "FLAG_OWN_CAR", "TARGET")

Серед клієнтів, які платять вчасно:

*   65.7% не мають авто
*   34.3% мають авто




Серед клієнтів із труднощами з оплатою:
* 69.5% не мають авто
* 30.5% мають авто

Різниця між категоріями є, але вона невелика. Клієнти без авто мають трохи вищу частку труднощів з оплатою, ніж клієнти з авто. Зв’язок із TARGET виглядає слабким.

# Кореляційний аналіз числових змінних

### Побудова кореляційної матриці для випадків із платіжними труднощами

In [ ]:
df1.select_dtypes(include=["int64", "float64"]).shape

Є 66 числових стовпчиків. Створимо кореляційну матрицб `corr` для кращого перегляду результатів

In [ ]:
corr_df1 = df1.select_dtypes(include=["int64", "float64"]).corr()

In [ ]:
corr_df1.head()

Створимо теплову карту для перегляду кореляцій вище 80% і 99,99%

In [ ]:
correlation_heatmap(corr_df1)
plt.show()

### Подивимось на 10 найкращих кореляцій для випадків із платіжними труднощами

In [ ]:
corr_df1[corr_df1 <= 0.99].unstack().sort_values(ascending=False).head(22)

Оскільки у нас є комбінації, що повторюються, дивлячись на наведену вище таблицю і видаляючи дублі, ми отримуємо топ-10 кореляцій, як показано нижче:

- AMT_GOODS_PRICE -              AMT_CREDIT                    0.98
- REGION_RATING_CLIENT -         REGION_RATING_CLIENT_W_CITY   0.96
- CNT_FAM_MEMBERS -              CNT_CHILDREN                  0.89
- DEF_60_CNT_SOCIAL_CIRCLE -     DEF_30_CNT_SOCIAL_CIRCLE      0.87
- REG_REGION_NOT_WORK_REGION -   LIVE_REGION_NOT_WORK_REGION   0.85
- LIVE_CITY_NOT_WORK_CITY -      REG_CITY_NOT_WORK_CITY        0.78
- AMT_ANNUITY -                  AMT_GOODS_PRICE               0.75
- AMT_ANNUITY -                  AMT_CREDIT                    0.75
- DAYS_EMPLOYED -                FLAG_DOCUMENT_6               0.62
- DAYS_BIRTH -                   DAYS_EMPLOYED                 0.58

### Побудова кореляційної матриці для випадків із вчасними платежеми

In [ ]:
df0.select_dtypes(include=["int64", "float64"]).shape

Є 66 числових стовпчиків. Створимо кореляційну матрицю `corr` для кращого перегляду результатів

In [ ]:
corr_df0 = df0.select_dtypes(include=["int64", "float64"]).corr()

In [ ]:
corr_df0.head()

Створимо теплову карту для перегляду кореляцій вище 80% і 99,99%

In [ ]:
correlation_heatmap(corr_df0)
plt.show()

### Подивимось на 10 найкращих кореляцій для вчасних платежів

In [ ]:
corr_df0[corr_df0 <= 0.99].unstack().sort_values(ascending=False).head(28)

Оскільки у нас є комбінації, що повторюються, дивлячись на наведену вище таблицю і видаляючи дублі, ми отримуємо топ-10 кореляцій, як показано нижче:

- AMT_GOODS_PRICE              AMT_CREDIT                    0.99
- REGION_RATING_CLIENT         REGION_RATING_CLIENT_W_CITY   0.95
- CNT_FAM_MEMBERS              CNT_CHILDREN                  0.88
- REG_REGION_NOT_WORK_REGION   LIVE_REGION_NOT_WORK_REGION   0.86
- DEF_30_CNT_SOCIAL_CIRCLE     DEF_60_CNT_SOCIAL_CIRCLE      0.86
- LIVE_CITY_NOT_WORK_CITY      REG_CITY_NOT_WORK_CITY        0.83
- AMT_ANNUITY                  AMT_GOODS_PRICE               0.78
- AMT_ANNUITY                  AMT_CREDIT                    0.77
- DAYS_BIRTH                   DAYS_EMPLOYED                 0.63
- DAYS_EMPLOYED                FLAG_DOCUMENT_6               0.60

### Порівняємо 10 найкращих кореляцій між випадками із платіжними труднощами та вчасними платежами

**Спостереження**

- Топ-10 кореляцій для Труднощів з оплатою та Вчасних платежів однакові, за винятком незначних відмінностей у відсотках кореляції
- Найвища кореляція для комбінації `AMT_GOODS_PRICE` та `AMT_CREDIT`.
- Для набору даних "Труднощі з оплатою" кореляція між `AMT_GOODS_PRICE` та `AMT_CREDIT` становить 0,98
- Для набору даних "Вчасні платежі" кореляція між `AMT_GOODS_PRICE` та `AMT_CREDIT` становить 0,99

# Одновимірний аналіз числових змінних

### Аналіз стовпця `AMT_CREDIT`

#### Пошук викидів в `AMT_CREDIT` при випадках із платіжними труднощами

Розрахунок IQR (Inter Quartile range)

In [ ]:
Min_value1, Max_value1 = iqr_bounds(df1["AMT_CREDIT"])
print(f"Мінімальне значення, до якого існують викиди: {Min_value1}")
print(f"Максимальне значення, після якого існують викиди: {Max_value1}")

Значення *до* (Q1 - 1.5 * IQR) та *після* (Q3 + 1.5 * IQR) є викидами.

#### Пошук викидів в `AMT_CREDIT` при випадках із вчасними оплатами

Розрахунок IQR (Inter Quartile range)

In [ ]:
Min_value0, Max_value0 = iqr_bounds(df0["AMT_CREDIT"])
print(f"Мінімальне значення, до якого існують викиди: {Min_value0}")
print(f"Максимальне значення, після якого існують викиди: {Max_value0}")

Значення *до* (Q1 - 1.5 * IQR) та *після* (Q3 + 1.5 * IQR) є викидами.

Видалення викидів і побудова діаграми розподілу

In [ ]:
kde_no_outliers(df0, df1, Max_value0, Max_value1, "AMT_CREDIT")

**Спостереження**

- Для `AMT_CREDIT` від 250000 до приблизно 650000 більше клієнтів мають труднощі з оплатою
- Для `AMT_CREDIT` > 750000 більше клієнтів, які вчасно здійснюють платежі

# ДЗ 5. Одновимірний аналіз числової колонки

За наведеним вище прикладом, проведіть одновимірний аналіз (виявлення викидів, їх усунення та побудова KDE  графіку) для числових змінних
- `YEARS_BIRTH`
- `AMT_GOODS_PRICE`
- `DAYS_EMPLOYED`

Опціонально:
- `CNT_CHILDREN`
- `AMT_INCOME_TOTAL`

Для цього винесіть функціонал для аналізу у функцію та викличіть функцію для кожної двійки змінних.

Зробіть висновки з аналізу.

In [ ]:
upper_bound_0 = outlier_range(df0, "YEARS_BIRTH")
upper_bound_1 = outlier_range(df1, "YEARS_BIRTH")
(upper_bound_0, upper_bound_1)

In [ ]:
kde_no_outliers(df0, df1, upper_bound_0, upper_bound_1, "YEARS_BIRTH")

Спостереження

Серед клієнтів із труднощами з оплатою більша кількість спостерігається у молодшому віці, приблизно до 42 років.
Серед клієнтів, які платять вчасно, розподіл більше зміщений у бік старшого віку, особливо після 42 років.

In [ ]:
upper_bound_for_amt_goods_price_0 = outlier_range(df0, "AMT_GOODS_PRICE")
upper_bound_for_amt_goods_price_1 = outlier_range(df1, "AMT_GOODS_PRICE")
(upper_bound_for_amt_goods_price_0, upper_bound_for_amt_goods_price_1)

In [ ]:
kde_no_outliers(
    df0,
    df1,
    upper_bound_for_amt_goods_price_0,
    upper_bound_for_amt_goods_price_1,
    "AMT_GOODS_PRICE",
)

Спостереження

*  У більшості діапазонів криві мають схожу форму
*  Для AMT_GOODS_PRICE від приблизно 350000 до приблизно 470000 більше клієнтів мають труднощі з оплатою
*  Для AMT_GOODS_PRICE > 470000 і  AMT_GOODS_PRICE < 350000 більше клієнтів, які вчасно здійснюють платежі, особливо це помітно біля ~700 000, ~900 000, ~1 100 000

In [ ]:
upper_bound_for_days_employed_0 = outlier_range(df0, "DAYS_EMPLOYED")
upper_bound_for_days_employed_1 = outlier_range(df1, "DAYS_EMPLOYED")
(upper_bound_for_days_employed_0, upper_bound_for_days_employed_1)

In [ ]:
kde_no_outliers(
    df0,
    df1,
    upper_bound_for_days_employed_0,
    upper_bound_for_days_employed_1,
    "DAYS_EMPLOYED",
)

Спостереження

Помітно, що люди, які працюють менше, частіше мають тркднощі з вчасною оплатою. А люди, які працюють більше п'яти з половиною років(2000 / 365) не мають проблем з оплатою.

# Двовимірний/Багатовимірний аналіз

## Неперервні vs неперервні змінні

### Аналіз стовпця `AMT_GOODS_PRICE` vs `AMT_CREDIT`

**Пошук викидів для ствопця `AMT_GOODS_PRICE ` для групи із платіжними труднощами**

In [ ]:
max_value1_AMT_GOODS_PRICE = outlier_range(df1, "AMT_GOODS_PRICE")
max_value1_AMT_GOODS_PRICE

**Пошук викидів для ствопця `AMT_CREDIT` для групи із платіжними труднощами**

In [ ]:
max_value1_AMT_CREDIT = outlier_range(df1, "AMT_CREDIT")
max_value1_AMT_CREDIT

**Пошук викидів для `AMT_GOODS_PRICE `для групи із вчасними оплатами**

In [ ]:
max_value0_AMT_GOODS_PRICE = outlier_range(df0, "AMT_GOODS_PRICE")
max_value0_AMT_GOODS_PRICE

**Пошук викидів для `AMT_CREDIT `для групи із вчасними оплатами**

In [ ]:
max_value0_AMT_CREDIT = outlier_range(df0, "AMT_CREDIT")
max_value0_AMT_CREDIT

Побудова діаграми розсіювання для порівняння з видаленими викидами

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.title("Payment difficulties")
sns.scatterplot(
    x=df1[df1["AMT_GOODS_PRICE"] < max_value1_AMT_GOODS_PRICE].AMT_GOODS_PRICE,
    y=df1[df1["AMT_CREDIT"] < max_value1_AMT_CREDIT].AMT_CREDIT,
    data=df1,
)
plt.ticklabel_format(style="plain", axis="x")
plt.ticklabel_format(style="plain", axis="y")

plt.subplot(1, 2, 2)
plt.title("On-Time Payments")
sns.scatterplot(
    x=df0[df0["AMT_GOODS_PRICE"] < max_value0_AMT_GOODS_PRICE].AMT_GOODS_PRICE,
    y=df0[df0["AMT_CREDIT"] < max_value0_AMT_CREDIT].AMT_CREDIT,
    data=df0,
)
plt.ticklabel_format(style="plain", axis="x")
plt.ticklabel_format(style="plain", axis="y")

plt.tight_layout(pad=4)
plt.show()

**Спостереження**
- AMT_GOODS_PRICE та AMT_CREDIT мають сильну позитивну кореляцію. Це означає, що зі збільшенням ціни товару зростає і сума кредиту

# ДЗ 6. Кореляційний аналіз для пари числових змінних

За наведеним вище прикладом, проведіть кореляційний аналіз для пар змінних
- AMT_ANNUITY і AMT_CREDIT

Опціонально:
- DAYS_EMPLOYED і AMT_INCOME_TOTAL
- AMT_CREDIT і DAYS_BIRTH

[Для цього винесіть функціонал для аналізу у функцію та викличіть функцію для кожної двійки змінних.](https://)
Зробіть висновок про наявність кореляції між змінними.

In [ ]:
plot_scatter_without_outliers_auto_bounds(
    df0=df0,
    df1=df1,
    x_column="AMT_ANNUITY",
    y_column="AMT_CREDIT",
)

Спостереження

AMT_ANNUITY та AMT_CREDIT також мають сильну позитивну кореляцію. Це означає, що чим більший періодичний платіж, тим більша сума кредиту.

## Неперервні та категоріальні змінні

### Аналіз стовпців `AMT_CREDIT` vs `NAME_EDUCATION_TYPE` vs `CODE_GENDER`

**Пошук викидів для `AMT_CREDIT `для групи із платіжними труднощами**

In [ ]:
max_value1_AMT_CREDIT = outlier_range(df1, "AMT_CREDIT")
max_value1_AMT_CREDIT

**Пошук викидів для `AMT_CREDIT ` для групи із вчасними платежами**

In [ ]:
max_value0_AMT_CREDIT = outlier_range(df0, "AMT_CREDIT")
max_value0_AMT_CREDIT

**Клієнт з платіжними труднощами**

In [ ]:
df1.groupby(by=["NAME_EDUCATION_TYPE", "CODE_GENDER"]).AMT_CREDIT.describe().head()

**Клієнт зі вчасними платежами**

In [ ]:
df0.groupby(by=["NAME_EDUCATION_TYPE", "CODE_GENDER"]).AMT_CREDIT.describe().head()

In [ ]:
bi_boxplot(
    df0,
    df1,
    "NAME_EDUCATION_TYPE",
    "AMT_CREDIT",
    max_value1_AMT_CREDIT,
    max_value0_AMT_CREDIT,
    "CODE_GENDER",
)

**Спостереження
- Клієнти з "академічним ступенем" мають широкий діапазон кредитів для своєчасних платежів, тоді як для клієнтів з проблемами з оплатою цей діапазон значно нижчий
- Якщо поглянути на зведену статистику, то клієнти з "вищою освітою" та проблемами з оплатою беруть середній та медіанний кредит у значно більшому діапазоні, ніж клієнти з вчасною оплатою.
- Клієнти-чоловіки з "вищою освітою" завжди сплачують кредит вчасно

 # ДЗ 7. Кореляційний аналіз між двома категоріальними змінними і числовою

Проведіть аналогічний кореляційний аналіз для трійок змінних

- AMT_INCOME_TOTAL vs NAME_FAMILY_STATUS vs CODE_GENDER

Опціонально - трійки які можна додатково проаналізувати:
- AMT_INCOME_TOTAL vs YEARS_BIRTH_CATEGORY vs NAME_HOUSING_TYPE
- AMT_GOODS_PRICE vs NAME_INCOME_TYPE vs CODE_GENDER
- AMT_INCOME_TOTAL vs OCCUPATION_TYPE vs CODE_GENDER

А ще можете також проаналізувати додатково до обовʼязкової свою трійку :)

Для цього винесіть функціонал для аналізу у функцію та викличіть функцію для кожної трійки змінних.

Зробіть висновок про наявність кореляції між змінними.


In [ ]:
numeric_vs_categorical_analysis(
    df0, df1, "AMT_INCOME_TOTAL", "NAME_FAMILY_STATUS", "CODE_GENDER"
)

Спостереження:


*   Помітно що у всіх NAME_FAMILY_STATUS чоловіки мають вищу медіану доходу ніж жінки
*   На правому графіку, де клієнти платять вчасно, боксплоти для деяких категорій виглядають трохи вищими, ніж у групі Payment Difficulties. Це може означати, що клієнти зі своєчасними платежами в середньому мають вищий або стабільніший дохід.
*  А також медіана дохожу трохи відрізняється в деяких категоріях(наприклад, спостерігається, що медіана доходу у чолловіків та жінок зі статусами - Widow майже не відрізняється, коли зі статосом Married мають більшу різницю в медіані доходу).



## Категоріальні та категоріальні змінні

### Аналіз стовпців `NAME_INCOME_TYPE` vs `CODE_GENDER`

In [ ]:
bi_countplot_target(df0, df1, "NAME_INCOME_TYPE", "CODE_GENDER")

**Спостереження**
- Клієнти категорії `Working` та `Male` мають більше труднощів з оплатою порівняно з тими, хто платить вчасно
- Клієнти категорії `Pensioner` та `Female` мають більше труднощів з оплатою порівняно з тими, хто платить вчасно
- Клієнти категорії `Businessman` та `Student` здійснюють платежі вчасно, хоча їхня історія невелика

# ДЗ 8. Аналіз взаємозалежностей між двома категоріальними змінними

Проведіть подібний до прикладу аналіз залежностей між категоріальними змінними для пар змінних

- NAME_EDUCATION_TYPE vs CODE_GENDER

Опціонально:
- NAME_FAMILY_STATUS vs OCCUPATION_TYPE
- OCCUPATION_TYPE vs NAME_CONTRACT_TYPE

Опишіть спостереження щодо того, чи є цікаві знахідки стосовно цільової змінної з цього аналізу.

In [ ]:
df["NAME_EDUCATION_TYPE"].value_counts()

In [ ]:
bi_countplot_target(df0, df1, "NAME_EDUCATION_TYPE", "CODE_GENDER")

Спостереження

На графіках видно, що клієнти з освітою Secondary / secondary special частіше представлені серед групи з труднощами з оплатою. Натомість клієнти з Higher education частіше зустрічаються серед тих, хто платить вчасно.
Клієнти категорії Academic degree здійснюють платежі вчасно, хоча їхня історія невелика





***
# ДЗ 9. Висновок з проведеного аналізу
Напишіть Ваш висновок з проведеного ананлізу, яким категоріям осіб Ви б видали кредит? Категорія може бути, наприклад, люди з такою-то освітою, з таким-то доходом, з таким-то досвідом роботи.

# Висновок: Категорії клієнтів, на яких слід орієнтуватися при наданні кредиту



Після проведеного аналізу я б більше схилялася надавати кредит клієнтам, які мають ознаки стабільнішої платоспроможності.

Найбільш надійними виглядають клієнти з вищим рівнем освіти, особливо з Higher education. Вони частіше зустрічаються серед тих, хто здійснює платежі вчасно. Категорія Academic degree також виглядає позитивно, але через малу кількість спостережень по ній не варто робити сильний висновок.

Також більш надійними виглядають клієнти з більшим досвідом роботи. За графіками видно, що люди, які працюють довше приблизно 5–6 років, частіше належать до групи клієнтів зі своєчасними платежами.

За фінансовими ознаками можна помітити, що клієнти з вищими значеннями AMT_GOODS_PRICE та AMT_CREDIT частіше представлені серед тих, хто платить вчасно. Особливо це помітно для вищих діапазонів AMT_GOODS_PRICE.

Отже, найкращими кандидатами для видачі кредиту виглядають клієнти з вищою освітою, стабільним і довшим досвідом роботи, достатнім доходом, вищою вартістю товару та без ознак високого ризику прострочення.